In [0]:
UC_CATALOG = "preethiworkspace_7405608993331015"
UC_SCHEMA  = "default"
VOLUME_NAME = "bank_fraud"

BASE = f"/Volumes/preethiworkspace_7405608993331015/default/bank_fraud"

paths = {
  "landing":      f"{BASE}/landing/transactions",
  "schemaLoc":    f"{BASE}/schema/transactions",
  "ckpt_bronze":  f"{BASE}/checkpoints/bronze",
  "ckpt_silver":  f"{BASE}/checkpoints/silver",
  "ckpt_gold":    f"{BASE}/checkpoints/gold",
  "reference":    f"{BASE}/reference"
}

tables = {
  "bronze": f"{UC_CATALOG}.{UC_SCHEMA}.bronze_transactions",
  "silver": f"{UC_CATALOG}.{UC_SCHEMA}.silver_transactions",
  "gold":   f"{UC_CATALOG}.{UC_SCHEMA}.gold_alerts",
  "stats":  f"{UC_CATALOG}.{UC_SCHEMA}.dim_category_stats"
}

# quick sanity check
display(dbutils.fs.ls(BASE))
paths, tables

In [0]:
from pyspark.sql.functions import current_timestamp

bronze_stream = (
  spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "csv")
    .option("cloudFiles.inferColumnTypes", "true")
    .option("cloudFiles.schemaLocation", paths["schemaLoc"])
    .option("header", "true")
    .load(paths["landing"])
    .withColumn("_bronze_ingest_ts", current_timestamp())
)

(
  bronze_stream.writeStream
    .format("delta")
    .option("checkpointLocation", paths["ckpt_bronze"])
    .outputMode("append")
    .trigger(availableNow=True).table(tables["bronze"])
)